# 04 – Build Combined Tables

**Purpose:** Build the daily and weekly tables used by the demand plan and demand sensing models.

**Inputs:** all loaded tables + `scope_item_profile` + `weather`
**Outputs:**
- `demand_daily`: item × store × day, 7,897,788 rows (sales, promo, holidays, oil, transactions, weather)
- `demand_weekly`: item × store × week, 1,136,349 rows (sales, promo days, holiday days)

**Key decisions:**
- Calendar spine per item × store from its own first sale; zero-sales days added (24% of days)
- Promo = FALSE on filled days (avoids target leakage)
- Holidays matched by national / regional / local level; transferred holidays ignored on their original date
- Oil forward-filled; missing transactions stay NULL
- Weekly table: known-in-advance signals only; partial weeks flagged (`days_in_week`)
- Row counts and unit totals reconcile at every step (no fan-out)

** Item × store pairs in scope

In [38]:
import duckdb
con = duckdb.connect('../data/favorita.duckdb')   # creates the file on first run
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

In [19]:
con.sql("""CREATE OR REPLACE TABLE scope_pairs AS
            WITH scope_pairs_int AS (
            SELECT 
            s.item_nbr,
            s.store_nbr,
            MIN(s.date) AS first_sale_date,
            MAX(s.date) AS last_sale_date,
            FROM sales s
            LEFT JOIN items i
                ON s.item_nbr = i.item_nbr
            WHERE s.date BETWEEN '2014-04-01' AND '2017-08-15'
            AND s.unit_sales >0
            AND s.store_nbr IN (44, 3, 17, 39, 50, 38, 11, 51, 34, 28, 40, 43, 27)
            AND i.family IN ('BEVERAGES','DAIRY','MEATS')
            GROUP BY s.item_nbr, s.store_nbr)
            SELECT 
            item_nbr,
            store_nbr,
            first_sale_date,
            last_sale_date
            FROM scope_pairs_int
            WHERE last_sale_date >= '2017-07-01'
            AND first_sale_date < '2017-01-01'""")
            

In [22]:
con.sql("""SELECT 
        COUNT(*) AS total_pair,
        SUM(CASE WHEN last_sale_date <'2017-07-01' THEN 1 END) AS pair_last_sale_before_2017_07_01
        FROM scope_pairs""").df()

,total_pair,pair_last_sale_before_2017_07_01
0,8133,NaN


** **Finding:** 8,133 item × store pairs in scope (out of a possible 10,569 = 813 items × 13 stores). The activity rules (first sale before 2017-01-01, last sale on/after 2017-07-01) are applied **per pair**, which also removes items delisted in a single store or launched there only in 2017, avoiding fake zeros and too-short histories.

** Daily calendar spine

In [23]:
# Build the spine: one row per pair per day, from the pair's first sale to the last data date
con.sql("""
CREATE OR REPLACE TABLE daily_spine AS
SELECT
    p.item_nbr,
    p.store_nbr,
    UNNEST(generate_series(p.first_sale_date, DATE '2017-08-15', INTERVAL 1 DAY))::DATE AS date
FROM scope_pairs p
""")

In [24]:
# Check 1: total rows in the spine
con.sql("SELECT COUNT(*) AS spine_rows FROM daily_spine").df()

,spine_rows
0,7897788


In [25]:
# Check 2: expected rows = sum of days per pair (first sale → 2017-08-15, inclusive)
con.sql("""
SELECT SUM(DATE '2017-08-15' - first_sale_date + 1) AS expected_rows
FROM scope_pairs
""").df()

,expected_rows
0,7897788.0


**Finding:** `daily_spine` has 7,897,788 rows, one per item × store × day from each pair's first sale date to 2017-08-15. It matches the expected count (sum of days per pair), proving the spine is complete with no gaps or duplicates. Each pair's calendar starts at its own first sale, so no zero rows exist before an item was available in a store.

** Attach sales and promotion

In [26]:
con.sql("""
CREATE OR REPLACE TABLE daily_base AS
SELECT
    sp.item_nbr,
    sp.store_nbr,
    sp.date,
    COALESCE(s.unit_sales, 0)                  AS unit_sales,          
    GREATEST(COALESCE(s.unit_sales, 0), 0)     AS unit_sales_clean,  
    s.item_nbr IS NOT NULL                     AS is_sales_recorded,  
    COALESCE(s.is_on_promotion, FALSE)         AS is_on_promotion 
FROM daily_spine sp
LEFT JOIN sales s
    ON  s.item_nbr  = sp.item_nbr
    AND s.store_nbr = sp.store_nbr
    AND s.date      = sp.date
""")

In [27]:
con.sql("""
SELECT
    COUNT(*)                                                    AS total_rows,
    COUNT(*) FILTER (WHERE NOT is_sales_recorded)               AS filled_zero_rows,
    COUNT(*) FILTER (WHERE NOT is_sales_recorded) * 100.0 / COUNT(*) AS pct_filled
FROM daily_base
""").df()

,total_rows,filled_zero_rows,pct_filled
0,7897788,1924405,24.36638


**Finding:** `daily_base` keeps all 7,897,788 spine rows (no fan-out). 1,924,405 rows (24.4%) had no sales record and were filled with 0 units, i.e. on about 1 in 4 days an item sold nothing in that store. Some zeros may reflect stockouts rather than zero demand (no inventory data available). Promotion is set to FALSE on filled days to avoid target leakage; `is_sales_recorded` keeps filled rows traceable.

In [28]:
con.sql("""
CREATE OR REPLACE TABLE store_holiday_flags AS
WITH active_holidays AS (
    SELECT *
    FROM holidays_events
    WHERE NOT is_transferred                      -- original date of a moved holiday = normal day
),
scoped_stores AS (
    SELECT DISTINCT store_nbr FROM scope_pairs     -- the 13 stores, not hardcoded
),
matched AS (
    SELECT st.store_nbr, h.date, h.holiday_type, h.locale
    FROM stores st
    JOIN scoped_stores ss ON ss.store_nbr = st.store_nbr
    JOIN active_holidays h
        ON  h.locale = 'National'
        OR (h.locale = 'Regional' AND h.locale_name = st.state)
        OR (h.locale = 'Local'    AND h.locale_name = st.city)
)
SELECT
    store_nbr,
    date,
    BOOL_OR(holiday_type IN ('Holiday','Transfer','Additional','Bridge') AND locale = 'National')               AS is_holiday_national,
    BOOL_OR(holiday_type IN ('Holiday','Transfer','Additional','Bridge') AND locale IN ('Regional','Local'))    AS is_holiday_local,
    BOOL_OR(holiday_type = 'Event')                                                                             AS is_event,
    BOOL_OR(holiday_type = 'Work Day')                                                                          AS is_work_day
FROM matched
GROUP BY store_nbr, date
""")

In [29]:
con.sql("""SELECT * FROM store_holiday_flags LIMIT 5""").df()

,store_nbr,date,is_holiday_national,is_holiday_local,is_event,is_work_day
0,28,2013-02-11,True,False,False,False
1,28,2013-05-01,True,False,False,False
2,28,2013-07-24,False,True,False,False
3,28,2013-12-21,True,False,False,False
4,28,2013-12-25,True,False,False,False


In [30]:
# 1. Must be unique per store + date (no fan-out later)
con.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT (store_nbr, date)) AS distinct_store_dates
FROM store_holiday_flags
""").df()

,total_rows,distinct_store_dates
0,2183,2183


In [31]:
# 2. Holiday and event days per store, Sierra vs Costa side by side
con.sql("""
SELECT f.store_nbr, st.city,
       COUNT(*) FILTER (WHERE is_holiday_national) AS national_days,
       COUNT(*) FILTER (WHERE is_holiday_local)    AS local_days,
       COUNT(*) FILTER (WHERE is_event)            AS event_days,
       COUNT(*) FILTER (WHERE is_work_day)         AS work_days
FROM store_holiday_flags f
JOIN stores st USING (store_nbr)
GROUP BY ALL
ORDER BY st.city, f.store_nbr
""").df()

,store_nbr,city,national_days,local_days,event_days,work_days
0,50,Ambato,102,12,55,5
1,11,Cayambe,102,6,55,5
2,39,Cuenca,102,6,55,5
3,27,Daule,102,0,55,5
4,43,Esmeraldas,102,6,55,5
5,28,Guayaquil,102,9,55,5
6,34,Guayaquil,102,9,55,5
7,51,Guayaquil,102,9,55,5
8,38,Loja,102,6,55,5
9,40,Machala,102,6,55,5


** **Finding:** `store_holiday_flags` has one row per store × date with any holiday or event (2,183 rows, unique). All 13 stores share 102 national holiday days, 55 event days and 5 work days. Local holidays differ by city (Quito/Ambato 12, Guayaquil 9, others 6), confirming city/state matching works. Daule has 0 local holidays because the source has none recorded for Daule or Guayas state (coverage gap, not a join error). Transferred holidays are ignored on their original date.

** Oil (forward-filled)

In [32]:
con.sql("""
CREATE OR REPLACE TABLE oil_daily AS
WITH calendar AS (
    SELECT d::DATE AS date
    FROM generate_series(DATE '2013-01-01', DATE '2017-08-15', INTERVAL 1 DAY) t(d)
)
SELECT
    c.date,
    o.oil_price_wti AS oil_price_raw,                          -- NULL on weekends, holidays, missing prices
    LAST_VALUE(o.oil_price_wti IGNORE NULLS) OVER (
        ORDER BY c.date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS oil_price_ffill                                        -- last known price up to that day (no future data)
FROM calendar c
LEFT JOIN oil o ON o.date = c.date
""")

In [33]:
con.sql("""
SELECT
    COUNT(*)                                                               AS total_days,
    COUNT(*) FILTER (WHERE oil_price_raw IS NULL)                          AS raw_null_days,
    COUNT(*) FILTER (WHERE oil_price_ffill IS NULL)                        AS ffill_null_days,
    COUNT(*) FILTER (WHERE oil_price_ffill IS NULL AND date >= '2014-04-01') AS ffill_null_in_scope
FROM oil_daily
""").df()

,total_days,raw_null_days,ffill_null_days,ffill_null_in_scope
0,1688,525,1,0


** **Finding:** `oil_daily` covers all 1,688 calendar days (2013-01-01 → 2017-08-15). 525 days had no raw price (weekends, market holidays, missing values); forward-fill carries the last known price into them using only past values, so there is no leakage. After filling, only 2013-01-01 is NULL (before scope); 0 NULLs from 2014-04-01 on. The base table stores the as-of-day price; horizon-specific lags are built in the modelling phase.

** Final combined table

In [35]:
con.sql("""
CREATE OR REPLACE TABLE demand_daily AS
SELECT
    -- keys and calendar
    b.date,
    DATE_TRUNC('week', b.date)::DATE           AS week_start,        -- Monday of that week (for the weekly roll-up)
    EXTRACT(isodow FROM b.date)::INTEGER       AS day_of_week,       -- 1 = Monday ... 7 = Sunday
    b.store_nbr,
    b.item_nbr,

    -- master data
    st.city, st.state, st.store_type, st.cluster,
    i.family, i.class, i.is_perishable,

    -- sales and promotion (11c)
    b.unit_sales,
    b.unit_sales_clean,
    b.is_sales_recorded,
    b.is_on_promotion,

    -- holidays (11d): no row = not a holiday
    COALESCE(h.is_holiday_national, FALSE)     AS is_holiday_national,
    COALESCE(h.is_holiday_local,    FALSE)     AS is_holiday_local,
    COALESCE(h.is_event,            FALSE)     AS is_event,
    COALESCE(h.is_work_day,         FALSE)     AS is_work_day,

    -- observed signals: as-of-day facts (lags built in the modelling phase)
    o.oil_price_ffill                          AS oil_price,
    t.no_of_transactions,                                            -- stays NULL if missing (missing ≠ 0)
    w.temp_max_deg_c,
    w.temp_min_deg_c,
    w.temp_mean_deg_c,
    w.precipitation_mm
FROM daily_base b
JOIN      items               i  ON i.item_nbr   = b.item_nbr
JOIN      stores              st ON st.store_nbr = b.store_nbr
LEFT JOIN store_holiday_flags h  ON h.store_nbr  = b.store_nbr AND h.date = b.date
LEFT JOIN oil_daily           o  ON o.date       = b.date
LEFT JOIN transactions        t  ON t.store_nbr  = b.store_nbr AND t.date = b.date
LEFT JOIN weather             w  ON w.city       = st.city     AND w.date = b.date
""")

In [36]:
con.sql("""
SELECT
    COUNT(*)                                              AS total_rows,
    COUNT(*) FILTER (WHERE oil_price IS NULL)             AS null_oil,
    COUNT(*) FILTER (WHERE temp_max_deg_c IS NULL)        AS null_weather,
    COUNT(*) FILTER (WHERE no_of_transactions IS NULL)    AS null_transactions,
    COUNT(*) FILTER (WHERE no_of_transactions IS NULL AND is_sales_recorded) AS null_txn_but_sales_exist
FROM demand_daily
""").df()

,total_rows,null_oil,null_weather,null_transactions,null_txn_but_sales_exist
0,7897788,0,0,56229,11007


In [37]:
con.sql("""SELECT * FROM demand_daily LIMIT 5""").df()

,date,week_start,day_of_week,store_nbr,item_nbr,city,state,store_type,cluster,family,...,is_holiday_national,is_holiday_local,is_event,is_work_day,oil_price,no_of_transactions,temp_max_deg_c,temp_min_deg_c,temp_mean_deg_c,precipitation_mm
0,2016-10-20,2016-10-17,4,38,1463801,Loja,Loja,D,4,BEVERAGES,...,False,False,False,False,50.31,997,23.4,14.5,18.8,0.0
1,2016-10-20,2016-10-17,4,38,1463805,Loja,Loja,D,4,BEVERAGES,...,False,False,False,False,50.31,997,23.4,14.5,18.8,0.0
2,2016-10-20,2016-10-17,4,38,1463807,Loja,Loja,D,4,BEVERAGES,...,False,False,False,False,50.31,997,23.4,14.5,18.8,0.0
3,2016-10-20,2016-10-17,4,38,1463808,Loja,Loja,D,4,BEVERAGES,...,False,False,False,False,50.31,997,23.4,14.5,18.8,0.0
4,2016-10-20,2016-10-17,4,38,1463812,Loja,Loja,D,4,BEVERAGES,...,False,False,False,False,50.31,997,23.4,14.5,18.8,0.0


** Weekly_table 

In [39]:
con.sql("""
CREATE OR REPLACE TABLE demand_weekly AS
SELECT
    week_start,
    store_nbr,
    item_nbr,
    city, state, store_type, cluster,
    family, class, is_perishable,
    COUNT(*)                                          AS days_in_week,          -- < 7 = partial week
    SUM(unit_sales)                                   AS unit_sales,
    SUM(unit_sales_clean)                             AS unit_sales_clean,
    COUNT(*) FILTER (WHERE is_on_promotion)           AS promo_days,
    COUNT(*) FILTER (WHERE is_holiday_national)       AS holiday_national_days,
    COUNT(*) FILTER (WHERE is_holiday_local)          AS holiday_local_days,
    COUNT(*) FILTER (WHERE is_event)                  AS event_days,
    COUNT(*) FILTER (WHERE is_work_day)               AS work_days
FROM demand_daily
GROUP BY ALL
""")

In [40]:
con.sql("""
SELECT
    (SELECT SUM(unit_sales) FROM demand_daily)                                    AS daily_total_units,
    (SELECT SUM(unit_sales) FROM demand_weekly)                                   AS weekly_total_units,
    (SELECT COUNT(*) FROM demand_weekly)                                          AS weekly_rows,
    (SELECT COUNT(DISTINCT (item_nbr, store_nbr, week_start)) FROM demand_weekly) AS distinct_keys,
    (SELECT COUNT(*) FROM demand_weekly WHERE days_in_week < 7)                   AS partial_week_rows
""").df()

,daily_total_units,weekly_total_units,weekly_rows,distinct_keys,partial_week_rows
0,7.632433e+07,7.632433e+07,1136349,1136349,15857


**Finding:** `demand_weekly` has 1,136,349 rows, one per item × store × week (Monday start), unique on its key. Total units reconcile exactly with `demand_daily` (76.32M). 15,857 rows (1.4%) are partial weeks (fewer than 7 days), mainly each pair's first week and the final week ending Tuesday 2017-08-15; `days_in_week` flags them so the modelling phase can exclude them. Only known-in-advance signals are included (promo days, holiday/event/work days); observed signals stay in the daily table for sensing.

In [41]:
con.close()     